# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadsammad42/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Ans: Rule: I will rank pages by refresh opportunity using Google Search Console impressions and average search position. Pages with high search visibility and weaker average positions receive the highest score because they already have search demand but may have room for improvement. The thresholds are calculated from the March 2026 data using the 75th percentile of impressions and the median of valid average positions.

Reason codes:

HIGH_VISIBILITY_POSITION_OPPORTUNITY — high impressions and a weaker average position.

MODERATE_OPPORTUNITY — only one of the two opportunity conditions is met.

LOW_OPPORTUNITY — neither condition is met.

In [11]:
%pip -q install duckdb

import os
import getpass
import duckdb
import pandas as pd

# Get Hugging Face token from Colab Secrets
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

# Create DuckDB connection
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 partition
FACT_MAR = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

print("DuckDB connection established.")
print("March 2026 warehouse partition ready.")

DuckDB connection established.
March 2026 warehouse partition ready.


In [12]:
# Part 1: Define the baseline scoring rule.
# Thresholds are supplied from the observed March 2026 data.

def assign_reason_and_score(
    impressions,
    avg_position,
    impression_high,
    position_opportunity
):
    """
    Assign one score and one reason code.

    Higher impressions indicate greater existing search visibility.
    A higher average position number indicates a weaker ranking position
    and therefore potentially more room for improvement.
    """

    if (
        impressions >= impression_high
        and avg_position >= position_opportunity
    ):
        return 100, "HIGH_VISIBILITY_POSITION_OPPORTUNITY"

    elif (
        impressions >= impression_high
        or avg_position >= position_opportunity
    ):
        return 60, "MODERATE_OPPORTUNITY"

    else:
        return 20, "LOW_OPPORTUNITY"

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Ans: Ranked queue: I apply the rule to the March 2026 data, calculate one opportunity score and one reason code for each eligible page, assign an action label, and rank pages from highest to lowest score. The queue uses only current-window information and excludes rows without GSC data or a valid average position.

In [13]:
# Part 2: Build the ranked queue and write the required CSV.

import pandas as pd
import os

# Load the March 2026 data needed by the baseline.
df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_avg_position
    FROM {FACT_MAR}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL
      AND gsc_avg_position IS NOT NULL
      AND gsc_avg_position > 0
""").df()

print(f"Eligible rows: {len(df)}")

# Data-driven thresholds from March 2026.
impression_high = df["gsc_impressions"].quantile(0.75)
position_opportunity = df["gsc_avg_position"].median()

print(f"High-impression threshold (75th percentile): {impression_high:.2f}")
print(f"Position-opportunity threshold (median): {position_opportunity:.2f}")

# Apply the baseline rule.
results = df.apply(
    lambda row: assign_reason_and_score(
        row["gsc_impressions"],
        row["gsc_avg_position"],
        impression_high,
        position_opportunity
    ),
    axis=1
)

df[["score", "reason_code"]] = pd.DataFrame(
    results.tolist(),
    index=df.index
)

# Convert scores into action labels.
df["action"] = df["score"].map({
    100: "REFRESH",
    60: "REVIEW",
    20: "MONITOR"
})

# Rank highest-priority pages first.
df = df.sort_values(
    by=["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = df.index + 1

# Select the final queue columns.
queue = df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action"
    ]
]

# Create output directory.
os.makedirs("work/outputs", exist_ok=True)

# Write the required baseline CSV.
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print(f"\nSaved ranked queue to: {output_path}")
print(f"Rows written: {len(queue)}")

# Display the top 10 for the next section.
queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible rows: 3447872
High-impression threshold (75th percentile): 66.00
Position-opportunity threshold (median): 8.00

Saved ranked queue to: work/outputs/baseline_action_score.csv
Rows written: 3447872


,rank,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_avg_position,score,reason_code,action
0,1,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,37368,8.613948,100,HIGH_VISIBILITY_POSITION_OPPORTUNITY,REFRESH
1,2,client_23a62021009f63c4,content_66288edeb93b7c4f,2026-03-29,24577,10.794239,100,HIGH_VISIBILITY_POSITION_OPPORTUNITY,REFRESH
2,3,client_23a62021009f63c4,content_66288edeb93b7c4f,2026-03-28,23542,11.112140,100,HIGH_VISIBILITY_POSITION_OPPORTUNITY,REFRESH
3,4,client_23a62021009f63c4,content_e943d753806d7af3,2026-03-28,15522,8.788429,100,HIGH_VISIBILITY_POSITION_OPPORTUNITY,REFRESH
4,5,client_23a62021009f63c4,content_e8a52cf3d5988c07,2026-03-09,15394,17.296804,100,HIGH_VISIBILITY_POSITION_OPPORTUNITY,REFRESH
5,6,client_23a62021009f63c4,content_e6df0936699f5b8f,2026-03-31,14682,25.035826,100,HIGH_VISIBILITY_POSITION_OPPORTUNITY,REFRESH
6,7,client_23a62021009f63c4,content_e8a52cf3d5988c07,2026-03-12,14138,16.244306,100,HIGH_VISIBILITY_POSITION_OPPORTUNITY,REFRESH
7,8,client_23a62021009f63c4,content_e8a52cf3d5988c07,2026-03-11,13910,16.680446,100,HIGH_VISIBILITY_POSITION_OPPORTUNITY,REFRESH
8,9,client_23a62021009f63c4,content_e8a52cf3d5988c07,2026-03-10,13060,16.536753,100,HIGH_VISIBILITY_POSITION_OPPORTUNITY,REFRESH
9,10,client_23a62021009f63c4,content_e8a52cf3d5988c07,2026-03-04,11347,16.566053,100,HIGH_VISIBILITY_POSITION_OPPORTUNITY,REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Ans: Top-20 review: I reviewed the 20 highest-ranked pages produced by the baseline rule. For each page, I record the assigned action, the reason code that triggered the score, a confidence note based on how strongly the observed signals support the rule, and what could make the recommendation wrong. These are decision-support recommendations rather than confirmed content-refresh requirements because the warehouse does not directly measure content quality, recent editorial changes, or the actual need for a refresh.

In [14]:
# Part 3: Top-20 review

top20 = queue.head(20).copy()

# Add a simple confidence note based on the rule score.
top20["confidence_note"] = top20["score"].map({
    100: "Higher confidence: both visibility and position conditions are met.",
    60: "Moderate confidence: only one opportunity condition is met.",
    20: "Lower confidence: neither opportunity condition is strongly met."
})

# Add a standard limitation for the rule.
top20["what_would_make_it_wrong"] = (
    "The page may not actually need a refresh; "
    "the rule cannot observe content quality, recent edits, "
    "or the reason behind its current search performance."
)

review = top20[
    [
        "rank",
        "content_hash_id",
        "score",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

review

,rank,content_hash_id,score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_945d6ff91386c817,100,REFRESH,HIGH_VISIBILITY_POSITION_OPPORTUNITY,Higher confidence: both visibility and positio...,The page may not actually need a refresh; the ...
1,2,content_66288edeb93b7c4f,100,REFRESH,HIGH_VISIBILITY_POSITION_OPPORTUNITY,Higher confidence: both visibility and positio...,The page may not actually need a refresh; the ...
2,3,content_66288edeb93b7c4f,100,REFRESH,HIGH_VISIBILITY_POSITION_OPPORTUNITY,Higher confidence: both visibility and positio...,The page may not actually need a refresh; the ...
3,4,content_e943d753806d7af3,100,REFRESH,HIGH_VISIBILITY_POSITION_OPPORTUNITY,Higher confidence: both visibility and positio...,The page may not actually need a refresh; the ...
4,5,content_e8a52cf3d5988c07,100,REFRESH,HIGH_VISIBILITY_POSITION_OPPORTUNITY,Higher confidence: both visibility and positio...,The page may not actually need a refresh; the ...
5,6,content_e6df0936699f5b8f,100,REFRESH,HIGH_VISIBILITY_POSITION_OPPORTUNITY,Higher confidence: both visibility and positio...,The page may not actually need a refresh; the ...
6,7,content_e8a52cf3d5988c07,100,REFRESH,HIGH_VISIBILITY_POSITION_OPPORTUNITY,Higher confidence: both visibility and positio...,The page may not actually need a refresh; the ...
7,8,content_e8a52cf3d5988c07,100,REFRESH,HIGH_VISIBILITY_POSITION_OPPORTUNITY,Higher confidence: both visibility and positio...,The page may not actually need a refresh; the ...
8,9,content_e8a52cf3d5988c07,100,REFRESH,HIGH_VISIBILITY_POSITION_OPPORTUNITY,Higher confidence: both visibility and positio...,The page may not actually need a refresh; the ...
9,10,content_e8a52cf3d5988c07,100,REFRESH,HIGH_VISIBILITY_POSITION_OPPORTUNITY,Higher confidence: both visibility and positio...,The page may not actually need a refresh; the ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Ans: Weak picks: Some recommendations may be wrong because the baseline only uses search impressions and average search position. A page may receive a high score because it has high visibility and a relatively weak position, even though it was recently updated or does not actually need a content refresh. The rule also cannot observe content quality, editorial plans, seasonality, or business priorities.

Leakage check: The baseline uses only March 2026 fields available at the decision moment: gsc_impressions and gsc_avg_position. It does not use future-month performance, outcome labels, product flags, or any label-derived field. June 2026 remains outside the feature window and is treated as a sealed future period.

In [15]:

# 1. Confirm the queue only contains March 2026 data.
print("Date range in baseline queue:")
print(queue["report_date"].min(), "to", queue["report_date"].max())

assert queue["report_date"].min() >= pd.Timestamp("2026-03-01")
assert queue["report_date"].max() <= pd.Timestamp("2026-03-31")

print("No future dates are present in the baseline queue.")


# 2. Confirm the queue contains only the intended baseline inputs.
allowed_input_columns = {
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_avg_position"
}

print("\nBaseline input columns:")
print(sorted(allowed_input_columns))

# These are the columns created by the rule, not input features.
derived_columns = {
    "rank",
    "score",
    "reason_code",
    "action"
}

print("\nDerived columns:")
print(sorted(derived_columns))


# 3. Confirm no obvious future/label-derived fields are in the queue.
forbidden_terms = [
    "label",
    "target",
    "outcome",
    "future",
    "refresh",
    "product_flag"
]

suspicious_columns = [
    col for col in queue.columns
    if any(term in col.lower() for term in forbidden_terms)
]

print("\nPotential leakage-related column names:")
print(suspicious_columns)

assert suspicious_columns == [], (
    f"Potential leakage-related columns found: {suspicious_columns}"
)

print("No label/future/product-flag columns are present.")

Date range in baseline queue:
2026-03-01 00:00:00 to 2026-03-31 00:00:00
No future dates are present in the baseline queue.

Baseline input columns:
['client_hash_id', 'content_hash_id', 'gsc_avg_position', 'gsc_impressions', 'report_date']

Derived columns:
['action', 'rank', 'reason_code', 'score']

Potential leakage-related column names:
[]
No label/future/product-flag columns are present.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.